In [0]:
dbutils.widgets.text("catalog", "oceanwatch_e2")
catalog = dbutils.widgets.get("catalog")
print("catalog:", catalog)

SOG_MAX = 50.0   # nudos; por encima es implausible para un mercante

tbl_bronze = f"{catalog}.bronze.ais_raw"
tbl_ref    = f"{catalog}.silver.ref_h3_superficie"
tbl_silver = f"{catalog}.silver.ais_posiciones"
tbl_quar   = f"{catalog}.silver.ais_cuarentena"
tbl_metr   = f"{catalog}.ops.calidad_metricas"
chk        = f"/Volumes/{catalog}/ops/checkpoints/silver_ais"

In [0]:
from pyspark.sql import functions as F, Window

CAUSAS = ["mmsi_invalido", "ts_invalido", "coord_fuera_rango", "posible_en_tierra", "sog_implausible"]
FLAGS  = ["sog_sin_dato", "cog_sin_dato", "heading_sin_dato", "cog_invalido", "heading_invalido",
          "imo_invalido", "fuera_extension"]
BRONZE_COLS = ['MMSI','BaseDateTime','LAT','LON','SOG','COG','Heading','VesselName','IMO','CallSign',
               'VesselType','Status','Length','Width','Draft','Cargo','TransceiverClass',
               '_rescued_data','_source_file','_ingested_at']

def num_o_nulo_si_cero(c, tipo="double"):
    v = F.col(c).cast(tipo)
    return F.when(v != 0, v)

def texto_o_nulo(c):
    v = F.trim(F.col(c))
    return F.when(v != "", v)

def transformar(df):
    mmsi_raw = F.trim(F.col("MMSI"))
    mmsi = F.when(mmsi_raw.rlike(r"^\d{7,9}$"), F.lpad(mmsi_raw, 9, "0"))   # E1 §2.4: ceros perdidos
    ts = F.col("BaseDateTime").cast("timestamp")
    lat, lon = F.col("LAT").cast("double"), F.col("LON").cast("double")
    sog, cog, hdg = (F.col(c).cast("double") for c in ["SOG", "COG", "Heading"])

    sog_sent = F.abs(sog - 102.3) < 0.05          # E1 §2.3: sentinels ITU-R M.1371
    cog_sent = F.abs(cog - 360.0) < 0.05
    hdg_sent = F.abs(hdg - 511) < 0.5
    cog_ok = (cog >= 0) & (cog < 360)
    hdg_ok = (hdg >= 0) & (hdg < 360)
    imo_ok = F.col("IMO").rlike(r"^IMO\d{7}$")
    coord_ok = lat.between(-90, 90) & lon.between(-180, 180)

    return df.select(
        mmsi.alias("mmsi"),
        ts.alias("ts"),
        F.to_date(ts).alias("fecha"),
        lat.alias("lat"), lon.alias("lon"),
        F.when(~sog_sent, sog).alias("sog"),
        F.when(cog_ok, cog).alias("cog"),
        F.when(hdg_ok, hdg).alias("heading"),
        texto_o_nulo("VesselName").alias("vessel_name"),
        F.when(imo_ok, F.col("IMO")).alias("imo"),
        texto_o_nulo("CallSign").alias("call_sign"),
        num_o_nulo_si_cero("VesselType", "int").alias("vessel_type"),
        F.col("Status").cast("int").alias("status"),
        num_o_nulo_si_cero("Length").alias("length"),
        num_o_nulo_si_cero("Width").alias("width"),
        num_o_nulo_si_cero("Draft").alias("draft"),
        num_o_nulo_si_cero("Cargo", "int").alias("cargo"),
        F.col("TransceiverClass").alias("transceiver_class"),
        F.when(coord_ok, F.expr("h3_longlatash3string(CAST(LON AS DOUBLE), CAST(LAT AS DOUBLE), 7)")).alias("h3_r7"),
        F.coalesce(mmsi.cast("long").between(200000000, 799999999), F.lit(False)).alias("es_buque"),
        # flags (no mandan a cuarentena)
        F.coalesce(sog_sent, F.lit(False)).alias("sog_sin_dato"),
        F.coalesce(cog_sent, F.lit(False)).alias("cog_sin_dato"),
        F.coalesce(hdg_sent, F.lit(False)).alias("heading_sin_dato"),
        F.coalesce(cog.isNotNull() & ~cog_ok & ~cog_sent, F.lit(False)).alias("cog_invalido"),
        F.coalesce(hdg.isNotNull() & ~hdg_ok & ~hdg_sent, F.lit(False)).alias("heading_invalido"),
        F.coalesce(F.col("IMO").isNotNull() & ~imo_ok, F.lit(False)).alias("imo_invalido"),
        F.coalesce((lon < -168) | (lon > -60) | (lat < 15) | (lat > 50), F.lit(False)).alias("fuera_extension"),
        coord_ok.alias("_coord_ok"),
        F.col("_source_file"), F.col("_ingested_at"),
        F.struct(*BRONZE_COLS).alias("raw"))

def causas_expr():
    return F.array_compact(F.array(
        F.when(F.col("mmsi").isNull(), F.lit("mmsi_invalido")),
        F.when(F.col("ts").isNull(), F.lit("ts_invalido")),
        F.when(~F.coalesce(F.col("_coord_ok"), F.lit(False)), F.lit("coord_fuera_rango")),
        F.when(F.col("en_tierra"), F.lit("posible_en_tierra")),
        F.when(F.col("sog") > SOG_MAX, F.lit("sog_implausible"))))

COLS_PLATA = ["mmsi", "ts", "fecha", "lat", "lon", "sog", "cog", "heading", "vessel_name", "imo", "call_sign",
              "vessel_type", "status", "length", "width", "draft", "cargo", "transceiver_class", "h3_r7",
              "es_buque"] + FLAGS + ["_source_file", "_ingested_at"]

In [0]:
def procesar(batch_df, batch_id):
    ss = batch_df.sparkSession

    ref = (ss.table(tbl_ref).filter("es_tierra")
             .select(F.col("h3").alias("h3_r7"), F.lit(True).alias("en_tierra")))
    df = (transformar(batch_df)
          .join(F.broadcast(ref), "h3_r7", "left")
          .withColumn("en_tierra", F.coalesce("en_tierra", F.lit(False)))
          .withColumn("causas", causas_expr())
          .withColumn("valido", F.size("causas") == 0))

    # Duplicados por (mmsi, ts) entre los válidos: se queda uno (E1 §2.6), orden determinístico
    w = Window.partitionBy("mmsi", "ts", "valido").orderBy("_source_file", "lat", "lon")
    df = df.withColumn("es_duplicado", F.col("valido") & (F.row_number().over(w) > 1))

    # 1. Plata: MERGE insert-only → idempotente y deduplica contra lo ya cargado
    plata = df.filter("valido AND NOT es_duplicado").select(*COLS_PLATA)
    if not ss.catalog.tableExists(tbl_silver):
        plata.limit(0).write.saveAsTable(tbl_silver)
        ss.sql(f"ALTER TABLE {tbl_silver} CLUSTER BY (fecha, mmsi)")
    plata.createOrReplaceTempView("lote_plata")
    m = ss.sql(f"""MERGE INTO {tbl_silver} s USING lote_plata b
                   ON s.fecha = b.fecha AND s.mmsi = b.mmsi AND s.ts = b.ts
                   WHEN NOT MATCHED THEN INSERT *""").first()
    insertadas = m["num_inserted_rows"]

    # 2. Cuarentena: filas crudas + causa documentada (append idempotente por batch)
    (df.filter("NOT valido")
       .select("raw.*", "causas", F.concat_ws(",", "causas").alias("causa"),
               F.lit(batch_id).alias("batch_id"), F.current_timestamp().alias("_quarantined_at"))
       .write.mode("append")
       .option("txnAppId", "silver_ais_cuarentena").option("txnVersion", batch_id)
       .saveAsTable(tbl_quar))

    # 3. Métricas de calidad por fecha (formato largo: fecha, metrica, valor)
    agg = df.groupBy("fecha").agg(
        F.count("*").alias("filas_entrada"),
        F.sum(F.col("valido").cast("int")).alias("validas"),
        F.sum(F.col("es_duplicado").cast("int")).alias("duplicados"),
        F.sum((~F.col("valido")).cast("int")).alias("cuarentena"),
        *[F.sum(F.array_contains("causas", c).cast("int")).alias(f"cuarentena_{c}") for c in CAUSAS],
        *[F.sum(F.col(c).cast("int")).alias(c) for c in FLAGS],
        F.sum((~F.col("es_buque")).cast("int")).alias("no_es_buque"))
    filas = agg.collect()                       # pocas filas (una por fecha)
    valores = [c for c in agg.columns if c != "fecha"]
    candidatas = sum((r["validas"] or 0) - (r["duplicados"] or 0) for r in filas)

    registros = [(r["fecha"], k, int(r[k] or 0)) for r in filas for k in valores]
    registros += [(None, "plata_insertadas", int(insertadas)),
                  (None, "duplicados_contra_historico", int(candidatas - insertadas))]
    metr = (ss.createDataFrame(registros, "fecha date, metrica string, valor long")
              .withColumn("batch_id", F.lit(batch_id))
              .withColumn("ts_proceso", F.current_timestamp()))
    (metr.write.mode("append")
         .option("txnAppId", "silver_ais_metricas").option("txnVersion", batch_id)
         .saveAsTable(tbl_metr))

In [0]:
q = (spark.readStream
     .option("maxBytesPerTrigger", "1g")
     .table(tbl_bronze)
     .writeStream
     .foreachBatch(procesar)
     .option("checkpointLocation", chk)
     .trigger(availableNow=True)
     .start())
q.awaitTermination()
print("stream terminado")

In [0]:
def show(df, n=50):
    print(df.limit(n).toPandas().to_string(index=False))

n_b = spark.table(tbl_bronze).count()
n_s = spark.table(tbl_silver).count()
n_q = spark.table(tbl_quar).count()
n_d = (spark.table(tbl_metr).filter("metrica IN ('duplicados', 'duplicados_contra_historico')")
       .agg(F.sum("valor")).first()[0] or 0)
print(f"bronce {n_b:,} = plata {n_s:,} + cuarentena {n_q:,} + duplicados {n_d:,}  →  diferencia: {n_b - n_s - n_q - n_d:,}")

print("\n-- Cuarentena por causa")
show(spark.table(tbl_quar).select(F.explode("causas").alias("causa")).groupBy("causa").count().orderBy(F.desc("count")))

print("\n-- Métricas totales")
show(spark.table(tbl_metr).groupBy("metrica").agg(F.sum("valor").alias("total")).orderBy("metrica"))